# 字符串处理与正则表达式：手机号 / 地址 / 姓名清洗实战

> 这是「数据分析从入门到精通」系列的第 13 篇。数值和日期处理完了，这篇来搞定文本数据——Pandas 的 str 系列函数加上正则表达式，文本清洗一步到位。

---

嗨，我是小荷。

文本数据是数据清洗里最"脏"的那类。格式千奇百怪：手机号前面加了 0 或 +86，地址有的写"北京市朝阳区"有的写"朝阳区北京"，姓名里夹着空格……

萧何当年整理秦朝户籍资料，估计也是这个感觉——"这人的名字怎么写的？这个地名是哪儿？" 幸好他有耐心，我们有 Pandas。

---

## Pandas 的 str 访问器

文本列的所有操作，都通过 `.str.` 来调用，**向量化处理，不需要写循环**。

In [1]:
import pandas as pd
import re

s = pd.Series(['  张三  ', '李四', '王 五', '赵六 '])


---

## 基础清洗

In [2]:
# 去除两端空白
s.str.strip()       # 去除首尾空格
s.str.lstrip()      # 去除左侧空格
s.str.rstrip()      # 去除右侧空格

# 去除中间空格
s.str.replace(' ', '')

# 大小写转换
emails = pd.Series(['Alice@Gmail.COM', 'BOB@163.com'])
emails.str.lower()   # 全小写
emails.str.upper()   # 全大写


0    ALICE@GMAIL.COM
1        BOB@163.COM
dtype: str

---

## 字符串查询

In [3]:
names = pd.Series(['张三丰', '李四', '王小五', '张无忌', '赵六'])

# 是否包含某字符串
names.str.contains('张')       # [True, False, False, True, False]

# 以某字符串开头/结尾
names.str.startswith('张')
names.str.endswith('忌')

# 字符串长度
names.str.len()

# 查找位置（-1 表示不存在）
names.str.find('三')


0    1
1   -1
2   -1
3   -1
4   -1
dtype: int64

---

## 字符串提取

In [4]:
# 截取子串（按位置）
s = pd.Series(['20240103', '20240215', '20241231'])
s.str[0:4]   # 取年份 ['2024', '2024', '2024']
s.str[-2:]   # 取日期

# 按分隔符分割
addresses = pd.Series(['北京市-朝阳区-望京街道', '上海市-浦东新区-陆家嘴'])
split_result = addresses.str.split('-', expand=True)
split_result.columns = ['城市', '区县', '街道']
print(split_result)
#     城市    区县    街道
# 0  北京市   朝阳区  望京街道
# 1  上海市  浦东新区   陆家嘴


    城市    区县    街道
0  北京市   朝阳区  望京街道
1  上海市  浦东新区   陆家嘴


---

## 正则表达式：文本清洗利器

正则不用全记，常用的几个模式够用了：

| 模式 | 含义 | 例子 |
|------|------|------|
| `\d` | 数字 0-9 | `\d+` 匹配一或多个数字 |
| `\w` | 字母/数字/下划线 | `\w+` 匹配单词 |
| `\s` | 空白字符 | `\s+` 匹配空格 |
| `[a-z]` | 字符范围 | `[a-zA-Z]+` 匹配英文单词 |
| `^` | 字符串开头 | `^1[3-9]` 手机号开头 |
| `$` | 字符串结尾 | `\.com$` 以.com结尾 |
| `.` | 任意字符 | `.*` 匹配任意字符串 |
| `{n}` | 重复n次 | `\d{4}` 4位数字 |

### 在 Pandas 里用正则

In [5]:
phones = pd.Series([
    '13812345678',
    '+8613912345678',
    '086-15612345678',
    '17888888888',
    '12345',           # 无效号码
    '138-1234-5678'
])

# 判断是否是合法手机号（1开头，11位，第2位3-9）
valid_pattern = r'^1[3-9]\d{9}$'
phones.str.match(valid_pattern)

# 提取手机号中的纯数字（去掉前缀）
phones_clean = phones.str.extract(r'(1[3-9]\d{9})', expand=False)
print(phones_clean)
# 0    13812345678
# 1    13912345678
# 2    15612345678
# 3    17888888888
# 4            NaN    ← 无效号码变 NaN
# 5    13812345678


0    13812345678
1    13912345678
2    15612345678
3    17888888888
4            NaN
5            NaN
dtype: str


---

## 实战：多类型文本数据清洗

In [8]:
import pandas as pd
import re

raw = pd.DataFrame({
    '姓名': ['  张 三  ', '李四', '王  五', None, '赵六'],
    '手机号': ['13812345678', '+86-139-1234-5678', '17888888888', '12345', '15600000000'],
    '地址': ['北京市朝阳区', '上海浦东', '广东省 广州市 天河区', '深圳南山', '杭州市西湖区'],
    '邮箱': ['ZHANG@Gmail.Com', 'li@163.com', None, 'invalid-email', 'zhao@qq.com']
})

df = raw.copy()

# 1. 姓名：去除空格，处理缺失
df['姓名'] = df['姓名'].str.replace(r'\s+', '', regex=True).str.strip()
df.fillna({'姓名':'未知'}, inplace=True)

# 2. 手机号：提取11位手机号
df['手机号_清洗'] = df['手机号'].str.extract(r'(1[3-9]\d{9})', expand=False)

# 3. 地址：去除多余空格，统一格式
df['地址'] = df['地址'].str.replace(r'\s+', '', regex=True)

# 4. 邮箱：转小写，标记无效格式
df['邮箱'] = df['邮箱'].str.lower()
email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
df['邮箱有效'] = df['邮箱'].str.match(email_pattern, na=False)

print(df[['姓名', '手机号_清洗', '地址', '邮箱', '邮箱有效']])

   姓名       手机号_清洗         地址               邮箱   邮箱有效
0  张三  13812345678     北京市朝阳区  zhang@gmail.com   True
1  李四          NaN       上海浦东       li@163.com   True
2  王五  17888888888  广东省广州市天河区              NaN  False
3  未知          NaN       深圳南山    invalid-email  False
4  赵六  15600000000     杭州市西湖区      zhao@qq.com   True


输出：
```
   姓名     手机号_清洗      地址         邮箱      邮箱有效
0  张三  13812345678   北京市朝阳区  zhang@gmail.com   True
1  李四  13912345678      上海浦东     li@163.com   True
2  王五  17888888888  广东省广州市天河区        None  False
3  未知          NaN       深圳南山  invalid-email  False
4  赵六  15600000000    杭州市西湖区    zhao@qq.com   True
```

---

## 常用 str 函数速查

| 函数 | 功能 |
|------|------|
| `str.strip()` | 去首尾空白 |
| `str.replace(old, new)` | 替换字符串 |
| `str.contains(pat)` | 是否包含 |
| `str.startswith(s)` | 是否以s开头 |
| `str.split(sep)` | 按分隔符切分 |
| `str.extract(pattern)` | 正则提取 |
| `str.match(pattern)` | 正则匹配（从头） |
| `str.findall(pattern)` | 找所有匹配项 |
| `str.len()` | 字符串长度 |
| `str.upper()` / `str.lower()` | 大小写转换 |
| `str.cat(sep)` | 字符串拼接 |

---

## 本篇小结

- Pandas 的 `.str.` 访问器让文本操作**向量化**，不用写循环
- 正则表达式是处理**格式不规范**文本的最强工具，常用模式记几个就够
- 实战清洗流程：去空格 → 提取有效内容 → 标记/处理无效值

---

## 课后练习

In [10]:
import pandas as pd

df = pd.DataFrame({
    '身份证号': ['110101199001011234', '31010219850315567X', '44010119920820abcd'],
    '薪资': ['8,500元/月', '¥12000/月', '年薪18万'],
    '工作地点': ['北京市 海淀区 中关村', '上 海 市 浦 东 新 区', '广州天河']
})

# 任务1：从身份证号提取出生年月日（格式：YYYY-MM-DD）
# 任务2：标记身份证号格式是否合法（18位，最后一位可以是X）
# 任务3：工作地点去除多余空格


In [11]:
# ========== 任务1：从身份证号提取出生年月日 ==========
print("\n" + "=" * 50)
print("任务1：提取出生年月日")
print("=" * 50)
# 身份证第7~14位是出生日期
df['出生日期'] = df['身份证号'].str.extract(r'(\d{4})(\d{2})(\d{2})').apply(
    lambda row: f'{row[0]}-{row[1]}-{row[2]}' if row[0] and row[0].isdigit() else '无法提取', axis=1
)
# 更稳健的写法：用正则替换
df['出生日期'] = df['身份证号'].str.replace(
    r'^\d{6}(\d{4})(\d{2})(\d{2}).*$', r'\1-\2-\3', regex=True
)
# 对于格式不合法的身份证，检查是否匹配
for i, row in df.iterrows():
    if not re.match(r'^\d{6}\d{8}', row['身份证号']):
        df.at[i, '出生日期'] = '无法提取'
print(df[['身份证号', '出生日期']])

# ========== 任务2：标记身份证号格式是否合法 ==========
print("\n" + "=" * 50)
print("任务2：身份证号合法性检查")
print("=" * 50)
# 合法：18位，前17位为数字，最后一位可以是数字或X
df['身份证合法'] = df['身份证号'].str.match(r'^\d{17}[\dX]$')
print(df[['身份证号', '身份证合法']])

# ========== 任务3：工作地点去除多余空格 ==========
print("\n" + "=" * 50)
print("任务3：去除多余空格")
print("=" * 50)
df['工作地点_清洗'] = df['工作地点'].str.replace(r'\s+', '', regex=True)
print("清洗前 → 清洗后:")
for i, row in df.iterrows():
    print(f"  '{row['工作地点']}' → '{row['工作地点_清洗']}'")


任务1：提取出生年月日
                 身份证号        出生日期
0  110101199001011234  1990-01-01
1  31010219850315567X  1985-03-15
2  44010119920820abcd  1992-08-20

任务2：身份证号合法性检查
                 身份证号  身份证合法
0  110101199001011234   True
1  31010219850315567X   True
2  44010119920820abcd  False

任务3：去除多余空格
清洗前 → 清洗后:
  '北京市 海淀区 中关村' → '北京市海淀区中关村'
  '上 海 市 浦 东 新 区' → '上海市浦东新区'
  '广州天河' → '广州天河'


评论区见 👀

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 14 篇：分组聚合 — groupby 从入门到精通**
>
> 按品类、按地区、按月份……数据分组是分析工作的核心。groupby 到底怎么用，为什么让人又爱又恨，下篇彻底讲明白。

---

👇 点「在看」，推给每天跟文本数据打架的人  
💬 评论区分享你遇到过最难清洗的文本格式  
⭐ 关注公众号，跟萧何迷妹一起搞定脏数据

---

*「数据分析从入门到精通」系列 · 第 13 篇*  
*上一篇：[第 12 篇：数据类型转换]*  
*下一篇：第 14 篇：分组聚合 — groupby 从入门到精通*